In [21]:
import httpx
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-0"

import math

http_client = httpx.Client(verify=False)
client = Anthropic(http_client=http_client)

def get_weather(location):
    # In a real app, you'd call a weather API here
    if "new york" in location.lower():
        return "22°C and Sunny"
    return "15°C and Cloudy"

def get_factorial(fact_number):
    # In a real app, you'd call a factorial method
    return str(math.factorial(fact_number))

# Define the tool schema for Claude
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather in a given location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA"
                }
            },
            "required": ["location"]
        }
    },
    {
        "name": "get_factorial",
        "description": "Get the factorial of given number",
        "input_schema": {
            "type": "object",
            "properties": {
                "number": {
                    "type": "integer",
                    "description": "The factorial  is "
                }
            },
            "required": ["number"]
        }    
    }
]

response = client.messages.create(
    model=model,
    max_tokens=1024,
    tools=tools,
    messages=[{"role": "user", "content": "What is the weather like in New York and What is factorial of 5?"}]
)

response = client.messages.create(
    model=model,
    max_tokens=1024,
    tools=tools,
    messages=[{"role": "user", "content": "What is the factorial of 5?"}]
)

if response.stop_reason == "tool_use":
    # Find the tool use block in the response
    tool_use = next(block for block in response.content if block.type == "tool_use")
    
    tool_name = tool_use.name
    tool_input = tool_use.input
    tool_use_id = tool_use.id

    # Execute the actual Python function
    if tool_name == "get_weather":
        result = get_weather(tool_input["location"])
    elif tool_name == "get_factorial":
        result=get_factorial(5)

        # Send the result back to Claude to get a final answer
        final_response = client.messages.create(
            model=model,
            max_tokens=1024,
            tools=tools,
            messages=[
                {"role": "user", "content": "What is the weather like in New York and What is factorial of 5?"},
                {"role": "assistant", "content": response.content},
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use_id,
                            "content": result,
                        }
                    ],
                },
            ],
        )
        print(final_response.content[0].text)
        #print(final_response.content)
        

C:\Users\61093123\AppData\Local\Temp\ipykernel_23936\356805172.py:55: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(
C:\Users\61093123\AppData\Local\Temp\ipykernel_23936\356805172.py:62: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(
C:\Users\61093123\AppData\Local\Temp\ipykernel_23936\356805172.py:84: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  final_

For the weather in New York, I need to clarify which New York you're referring to. Could you please specify "New York, NY" (New York City) or provide the specific city and state? The weather function requires the city and state format (e.g., "New York, NY" or "Albany, NY").

Meanwhile, the factorial of 5 is **120** (5 × 4 × 3 × 2 × 1 = 120).
